# Comm-MAPOCA — Blind Walker (fresh run, 30M steps)

Trains the Blind Walker environment (2 Oracle watchers + 1 color-blind walker,
emergent messages, random goal colors) with the `comm_mapoca` trainer, FRESH
from step 0 to `max_steps: 30M` (config: `config/comm_mapoca/CommOracle.yaml`).

## Session checklist (set in the right panel BEFORE running)
1. **Add Input** -> Your Datasets -> `blindwalker-data` (contains `BWComm_Linux.zip`, auto-extracted)
2. **Accelerator: None** (workload is CPU-bound; saves the 30h/week GPU quota)
3. **Internet: On** (needed for git clone + pip)
4. **Add-ons -> Secrets** -> attach `GITHUB_TOKEN` (fine-grained PAT, Contents: Read-only)

Run cells top to bottom. For a long unattended run: **Save Version -> Save & Run All (Commit)**.
30M fresh steps likely exceeds one background session, so plan a `--resume` chain:
next session, add the previous version's Output as an Input, copy its `results/`
into `/kaggle/working/`, and append `--resume` to the training command.
Checkpoints save every 500k steps - a cutoff never loses more than that.

In [ ]:
# Cell 1 - install uv, get exact Python 3.10.12 (ML-Agents R23 requirement), clone the repo
!pip -q install uv
!uv python install 3.10.12
!rm -rf repo
from kaggle_secrets import UserSecretsClient
import os
os.environ['GT'] = UserSecretsClient().get_secret("GITHUB_TOKEN")
!git clone https://$GT@github.com/Lihaj/4th-Year-Research-COMM-MAPOCA.git repo

In [ ]:
# Cell 2 - venv + the proven-working stack (matches the local machine exactly)
# setuptools<81 restores pkg_resources; torch pinned to 2.5.1 (avoids the
# onnxscript export path of newer torch). CPU wheels are fine: Accelerator=None.
!uv venv --python 3.10.12 mlagents-venv
!uv pip install --python mlagents-venv -e repo/ml-agents-envs -e repo/ml-agents
!uv pip install --python mlagents-venv "setuptools<81"
!uv pip install --python mlagents-venv torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1

In [ ]:
# Cell 2b - verify the dataset's real folder nesting BEFORE the copy step
!ls -R /kaggle/input/datasets/lihajwicky/blindwalker-data | head -40

In [ ]:
# Cell 3 - copy the game build into the working dir
# NOTE: path assumes single-level nesting (BWComm_Linux/ directly contains the
# .x86_64). If Cell 2b shows BWComm_Linux/BWComm_Linux/, add the extra level here.
DATA = "/kaggle/input/datasets/lihajwicky/blindwalker-data"
!mkdir -p env results
!cp -r "{DATA}/BWComm_Linux/." env/
!chmod +x env/BWComm.x86_64

In [ ]:
# Cell 4 - verify the binary is a valid Linux executable and boots headless
!file env/BWComm.x86_64
!timeout 15 ./env/BWComm.x86_64 -batchmode -nographics -logFile - || echo "exit code: $?"
# exit code 124 = killed by timeout after running fine (GOOD). Instant crash = problem.
# Look for the two startup lines: '[Oracle Watcher A Red] watching color: RED' / 'BLUE'.

In [ ]:
# Cell 5 - train Blind Walker with comm_mapoca, FRESH run to 30M
# For a later session after a background-limit cutoff: add previous version's
# Output as an Input, copy its results/ into /kaggle/working/, append --resume.
!mlagents-venv/bin/mlagents-learn repo/config/comm_mapoca/CommOracle.yaml \
  --env=env/BWComm.x86_64 --no-graphics --num-envs=2 \
  --run-id=blindwalker_comm_cloud_s1

In [ ]:
# Cell 6 - package results for download (runs after training stops)
!zip -r -q results_out.zip results/blindwalker_comm_cloud_s1
!ls -la results_out.zip

In [ ]:
# Cell 7 - OPTIONAL: live TensorBoard peek (interactive sessions only)
%load_ext tensorboard
%tensorboard --logdir results